# Full-frame performance measurements

The primary source is a centered AB 20 flat-$f_
u$ continuum. Every arm and the full configured wavelength range are included. Setup, observation and readout are timed separately. `profile=True` records Yappi wall-clock profiles, including profiler overhead; set it to `False` for interactive timings. For independent peak-memory measurements, restart this notebook's kernel between the ZShooter and X-shooter sections and rerun the imports.

The saved CSV retains the earlier baseline and intermediate measurements. `shared_final` uses the adopted branches. The historical X-shooter process peak follows earlier ZShooter runs and is cumulative. Sky downloads affect setup time; the X-shooter section uses an optional local ESO cache with the displayed fixed query parameters. The [comparison notebook](../../notebooks/02_Flat_AB_Comparison.ipynb) separately times its common-sky observations.


In [1]:
from pathlib import Path
from time import perf_counter
import resource
import sys
import numpy as np
import pandas as pd
import yappi
import scopesim as sim
from scopesim.source.source_templates import ab_spectrum
from zs_scopesim_tools.helpers import configure_irdb_path, warning_prevent_sync_alt_ra_dec, disable_scopesim_progress_bars, set_scopesim_log_level

workspace = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'env/zshooter-stack.toml').exists())
irdb = Path(configure_irdb_path())
set_scopesim_log_level('warning')
output = workspace / 'validation/xshooter/data'
measurements = pd.read_csv(output / 'full_frame_timings.csv').to_dict('records')
profile = True
source = sim.Source(x=[0.], y=[0.], ref=[0], weight=[1.], spectra=[ab_spectrum(mag=20)])
print(sys.executable, sim.__version__, sim.__file__)

/Users/jibailey/miniforge3/envs/zssim/bin/python 0.12.1.dev6 /Users/jibailey/src/scopesim/scopesim/__init__.py


In [2]:
instrument = 'ZShooter_v2'
label = 'shared_final'
properties = {'!OBS.airmass': 1.3, '!OBS.seeing': .7, '!OBS.brightness': 'dark', '!OBS.mjdobs': 60310.25}
properties.update({'!INST.vis_curr_slit': .7, '!INST.nir_curr_slit': .7, '!INST.include_diffuse_emissivity': False, '!ATMO.skycalc_timeout': 60.})
for arm in ('blue', 'green', 'red', 'yj', 'h', 'k'):
    properties[f'!OBS.dit_{arm}'] = 300.
    properties[f'!OBS.ndit_{arm}'] = 12
started = perf_counter()
cmds = sim.UserCommands(use_instrument=instrument, set_modes=['SPEC'], properties=properties)
warning_prevent_sync_alt_ra_dec(cmds)
train = sim.OpticalTrain(cmds)
setup_s = perf_counter() - started
print(f'{instrument}: setup {setup_s:.2f} s')


py.warnings - WARNING: /Users/jibailey/src/scopesim/scopesim/effects/spectral_trace_list.py:170: DeprecationWarning: The fov_grid method is deprecated and will be removed in a future release.
  fov = thetrace.fov_grid()

py.warnings - WARNING: /Users/jibailey/src/scopesim/scopesim/effects/spectral_trace_list.py:225: DeprecationWarning: The fov_grid method is deprecated and will be removed in a future release.
  vol = spt.fov_grid()

ZShooter_v2: setup 8.55 s


In [3]:
yappi.clear_stats()
yappi.set_clock_type('wall')
if profile:
    yappi.start()
started = perf_counter()
with disable_scopesim_progress_bars(True):
    train.observe(source)
observed = perf_counter()
readouts = train.readout()
finished = perf_counter()
if profile:
    yappi.stop()
    stats = yappi.get_func_stats()
    stats.sort('ttot', 'desc')
    with (output / f'{label}_{instrument}_yappi.txt').open('w') as stream:
        stats.print_all(out=stream, columns={0: ('name', 140), 1: ('ncall', 12), 2: ('tsub', 12), 3: ('ttot', 12)})
measurements.append({'instrument': instrument, 'label': label, 'profiled': profile, 'setup_s': setup_s, 'observe_s': observed - started, 'readout_s': finished - observed, 'peak_process_gib': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024**3})
pd.DataFrame(measurements).to_csv(output / 'full_frame_timings.csv', index=False)
print([(h[1].data.shape, np.isfinite(h[1].data).all()) for h in readouts])
display(pd.DataFrame(measurements))

py.warnings - WARNING: /Users/jibailey/src/scopesim/scopesim/effects/spectral_trace_list.py:225: DeprecationWarning: The fov_grid method is deprecated and will be removed in a future release.
  vol = spt.fov_grid()

[((4096, 2048), np.True_), ((4096, 2048), np.True_), ((4096, 2048), np.True_), ((2048, 2048), np.True_), ((2048, 2048), np.True_), ((2048, 2048), np.True_)]


,instrument,label,profiled,setup_s,observe_s,readout_s,peak_process_gib
0,ZShooter_v2,before_upstream,True,64.076229,70.069178,1.528787,6.245590
1,ZShooter_v2,before_upstream,True,64.076229,68.716845,1.551403,7.379410
2,XSHOOTER,before_upstream,True,1.132821,590.347333,1.146228,35.741592
3,ZShooter_v2,shared_spatial_first,True,8.305929,74.193853,1.656233,5.894028
4,XSHOOTER,shared_cached_shifts,True,1.267620,97.764302,0.887620,9.866638
5,ZShooter_v2,shared_final,True,8.547715,60.892919,1.680166,5.118393


In [2]:
instrument = 'XSHOOTER'
label = 'shared_final'
# This optional cache contains the same ESO query parameters as the live request.
sky_file = workspace / 'validation/xshooter/cache/paranal_x1p3_pwv2p5_r200000.fits'
properties = {'!OBS.airmass': 1.3, '!OBS.seeing': .7, '!OBS.mjdobs': 60310.25, '!ATMO.local_sky_file': str(sky_file) if sky_file.exists() else False}
for arm in ('uvb', 'vis', 'nir'):
    properties[f'!OBS.dit_{arm}'] = 300.
    properties[f'!OBS.ndit_{arm}'] = 12
started = perf_counter()
cmds = sim.UserCommands(use_instrument=instrument, set_modes=['SPEC'], properties=properties)
train = sim.OpticalTrain(cmds)
setup_s = perf_counter() - started
print(f'{instrument}: setup {setup_s:.2f} s; local sky cache: {sky_file.exists()}')


py.warnings - WARNING: /Users/jibailey/src/scopesim/scopesim/effects/spectral_trace_list.py:170: DeprecationWarning: The fov_grid method is deprecated and will be removed in a future release.
  fov = thetrace.fov_grid()

XSHOOTER: setup 1.37 s; local sky cache: True


In [3]:
yappi.clear_stats()
yappi.set_clock_type('wall')
yappi.start()
started = perf_counter()
with disable_scopesim_progress_bars(True):
    train.observe(source)
observed = perf_counter()
readouts = train.readout()
finished = perf_counter()
yappi.stop()
stats = yappi.get_func_stats()
stats.sort('ttot', 'desc')
with (output / f'{label}_{instrument}_yappi.txt').open('w') as stream:
    stats.print_all(out=stream, columns={0: ('name', 140), 1: ('ncall', 12), 2: ('tsub', 12), 3: ('ttot', 12)})
measurements.append({'instrument': instrument, 'label': label, 'profiled': True, 'setup_s': setup_s, 'observe_s': observed - started, 'readout_s': finished - observed, 'peak_process_gib': resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024**3})
pd.DataFrame(measurements).to_csv(output / 'full_frame_timings.csv', index=False)
print([(h[1].data.shape, np.isfinite(h[1].data).all()) for h in readouts])
display(pd.DataFrame(measurements))

py.warnings - WARNING: /Users/jibailey/src/scopesim/scopesim/effects/spectral_trace_list.py:225: DeprecationWarning: The fov_grid method is deprecated and will be removed in a future release.
  vol = spt.fov_grid()

[((3000, 2048), np.True_), ((4000, 2048), np.True_), ((1100, 2048), np.True_)]


,instrument,label,profiled,setup_s,observe_s,readout_s,peak_process_gib
0,ZShooter_v2,before_upstream,True,64.076229,70.069178,1.528787,6.245590
1,ZShooter_v2,before_upstream,True,64.076229,68.716845,1.551403,7.379410
2,XSHOOTER,before_upstream,True,1.132821,590.347333,1.146228,35.741592
3,ZShooter_v2,shared_spatial_first,True,8.305929,74.193853,1.656233,5.894028
4,XSHOOTER,shared_cached_shifts,True,1.267620,97.764302,0.887620,9.866638
5,ZShooter_v2,shared_final,True,8.547715,60.892919,1.680166,5.118393
6,XSHOOTER,shared_final,True,1.373102,74.404743,0.876663,9.322647
